# H4 · Sector × country type: does the best sector depend on the kind of country?

> **Hypothesis H4:** Which sectors earn more or less depends on the type of country: sector differences in ROA are **not the same** across country types.

| | |
|---|---|
| **Why we ask** | Sector explains little of ROA overall, but sectors have very different business models (banks, utilities, retail, tech). If country conditions matter, they may matter *differently* for different sectors, which an average effect would hide. |
| **Prediction** | The model with sector × country-type interactions explains **more** ROA variance than the additive model (sector + type). |
| **Data** | Yahoo company-years with a sector (excluding the 'Unknown' group). *Country types* = k-means clusters of the 35 country features (principal components covering 80% of variance; 3 to 5 clusters; every cluster at least 3 countries). |
| **Primary test (fixed before looking at the result)** | F-statistic of the sector × type interaction terms (on top of sector + type + relative size + fiscal year). The p-value comes from a **permutation test**: the type labels are shuffled across the 34 *countries* 1,999 times, which respects that all firms in a country share one type. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. *H4 predicts no direction* (any interaction counts), so the sign condition is always met. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Country types
The same clustering as notebook 03. Types are numbered from the highest GDP per person (1) downwards.

In [2]:
lab, info = utils.country_clusters(profile)
fy["ctype"] = fy.iso3.map(lab)
members = pd.DataFrame({"type": lab, "country": names}).sort_values(["type", "country"])
print(f"{info['k']} types from {info['n_components']} principal components; silhouette by k: " + ", ".join(f"k={k}: {v:.2f}" for k, v in info["silhouette"].items()))
prof_orig = profile.copy(); prof_orig["gdp_per_capita_ppp"] = np.exp(prof_orig.gdp_per_capita_ppp)
key = ["gdp_per_capita_ppp", "rule_of_law", "unemployment", "inflation", "trade_gdp", "rd_gdp", "internet_users", "pop_65plus"]
t = prof_orig[key].groupby(lab).mean().round(2); t.insert(0, "countries", lab.value_counts().sort_index())
t.insert(1, "member countries", members.groupby("type").country.apply(lambda s: ", ".join(s)))
t

4 types from 7 principal components; silhouette by k: k=3: 0.22, k=4: 0.25, k=5: 0.25


,countries,member countries,gdp_per_capita_ppp,rule_of_law,unemployment,inflation,trade_gdp,rd_gdp,internet_users,pop_65plus
1,3,"Ireland, Luxembourg, Singapore","123,348.250",1.590,4.810,3.060,322.080,1.440,94.960,13.910
2,13,"Australia, Austria, Belgium, Canada, Denmark, ...","59,880.440",1.560,5.250,3.320,87.890,3.050,92.510,20.200
3,13,"Chile, Czechia, Estonia, Greece, Hungary, Ital...","38,395.630",0.750,9.100,5.680,105.030,1.210,84.570,19.120
4,5,"Brazil, China, India, Indonesia, Mexico","16,164.830",-0.430,5.830,4.240,46.790,1.280,67.530,8.810


In [3]:
mp = pd.DataFrame({"iso3": lab.index, "type": lab.astype(str).values, "country": names[lab.index].values})
fig = px.choropleth(mp, locations="iso3", color="type", hover_name="country", category_orders={"type": [str(i) for i in range(1, info["k"] + 1)]},
                    color_discrete_sequence=px.colors.qualitative.Set2, title="Country types (1 = highest GDP per person)")
fig.update_geos(fitbounds="locations", projection_type="natural earth"); fig.update_layout(height=440, margin=dict(l=0, r=0, t=50, b=0)); fig.show()

## 2 · ROA by sector and country type
Median ROA (pp) per cell, shown only where the cell has at least 15 company-years from at least 3 countries. The second map removes the sector's overall level and the type's overall level, leaving the **interaction**: green = this sector does better in this type of country than the two averages would predict.

In [4]:
d = fy[(fy.sector != "Unknown")].dropna(subset=["roa_pp", "ctype", "size_rank"]).copy()
d["ctype"] = d.ctype.astype(int)
cell = d.groupby(["sector", "ctype"]).agg(med=("roa_pp", "median"), n=("roa_pp", "size"), countries=("iso3", "nunique"))
cell = cell[(cell.n >= 15) & (cell.countries >= 3)]
sec_med, typ_med, all_med = d.groupby("sector").roa_pp.median(), d.groupby("ctype").roa_pp.median(), d.roa_pp.median()
cell["interaction"] = cell.med - cell.index.get_level_values(0).map(sec_med) - (cell.index.get_level_values(1).map(typ_med) - all_med)
m1 = cell.med.unstack().reindex(sec_med.sort_values().index)
m2 = cell.interaction.unstack().reindex(sec_med.sort_values().index)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Median ROA (pp)", "Interaction: cell minus sector level minus type level (pp)"), horizontal_spacing=0.18)
fig.add_trace(go.Heatmap(z=m1.values, x=[f"type {c}" for c in m1.columns], y=m1.index, colorscale="Blues", text=m1.round(1).values, texttemplate="%{text}", colorbar=dict(x=0.43, len=.8)), row=1, col=1)
fig.add_trace(go.Heatmap(z=m2.values, x=[f"type {c}" for c in m2.columns], y=m2.index, colorscale="RdYlGn", zmid=0, text=m2.round(1).values, texttemplate="%{text}", colorbar=dict(x=1.0, len=.8)), row=1, col=2)
fig.update_layout(height=520); fig.show()
cell.assign(**{"cell": cell.index.map(lambda t: f"{t[0]} / type {t[1]}")}).sort_values("interaction")[["cell", "med", "interaction", "n", "countries"]].round(2).pipe(lambda x: pd.concat([x.head(5), x.tail(5)]))

cell   med  interaction    n  countries
sector             ctype                                                                
Energy             1                  Energy / type 1 2.550       -2.130   48          3
Consumer Cyclical  4       Consumer Cyclical / type 4 4.700       -1.270  201          5
Technology         3              Technology / type 3 4.510       -1.250  203         10
Healthcare         1              Healthcare / type 1 2.950       -1.210   79          3
                   3              Healthcare / type 3 3.170       -1.110  183         10
Utilities          3               Utilities / type 3 3.300        0.880  257         11
Real Estate        3             Real Estate / type 3 3.640        1.040  264         11
Technology         4              Technology / type 4 8.370        1.440  116          4
Healthcare         4              Healthcare / type 4 7.170        1.730  156          5
Consumer Defensive 4      Consumer Defensive / type 4 8.150        1.950  205          5

## 3 · Primary test: is the interaction more than chance?
The additive model (sector + type + size rank + fiscal year) is compared with the model that adds all sector × type terms. The **F-statistic** measures how much better the interaction model fits. To test it fairly, the type labels are **shuffled across countries** (keeping each country's firms together) 1,999 times to build the F-statistic one would see by chance.

In [5]:
dd = d.reset_index(drop=True)
Sd = pd.get_dummies(dd.sector, drop_first=True).astype(float).values
Yd = pd.get_dummies(dd.fiscal_year, drop_first=True).astype(float).values
base_cols = np.column_stack([np.ones(len(dd)), Sd, dd.size_rank.values, Yd])
y = dd.roa_pp.values
countries_ = dd.iso3.values
def fstat(country_type):
    t = pd.Series(country_type)[countries_].values
    T = pd.get_dummies(pd.Series(t), drop_first=True).astype(float).values
    Xa = np.column_stack([base_cols, T])
    inter = np.column_stack([Sd[:, [j]] * T[:, [l]] for j in range(Sd.shape[1]) for l in range(T.shape[1])])
    inter = inter[:, inter.std(axis=0) > 0]
    Xi = np.column_stack([Xa, inter])
    rss_a = np.sum((y - Xa @ np.linalg.lstsq(Xa, y, rcond=None)[0]) ** 2)
    rss_i = np.sum((y - Xi @ np.linalg.lstsq(Xi, y, rcond=None)[0]) ** 2)
    df1, df2 = Xi.shape[1] - Xa.shape[1], len(y) - Xi.shape[1]
    adj = lambda rss, k: 1 - (rss / (len(y) - k)) / (np.sum((y - y.mean()) ** 2) / (len(y) - 1))
    return ((rss_a - rss_i) / df1) / (rss_i / df2), adj(rss_i, Xi.shape[1]) - adj(rss_a, Xa.shape[1]), df1
labs = pd.Series(lab.values, index=lab.index)
F_obs, dR2, df1 = fstat(labs)
rng = np.random.default_rng(0)
null = np.array([fstat(pd.Series(rng.permutation(labs.values), index=labs.index))[0] for _ in range(1999)])
p_perm = float((np.sum(null >= F_obs) + 1) / (len(null) + 1))
print(f"{len(dd):,} company-years, {dd.iso3.nunique()} countries, {info['k']} types, {df1} interaction terms.")
print(f"F = {F_obs:.2f}; extra adjusted R² from the interaction = {dR2:+.4f}; permutation p = {p_perm:.4f}. (Null F: median {np.median(null):.2f}, 95th percentile {np.percentile(null, 95):.2f})")
px.histogram(x=null, nbins=50, title="F-statistic when country types are shuffled across countries (1,999 shuffles)", labels={"x": "F"}, height=340).add_vline(x=F_obs, line_color="red", annotation_text="observed")

10,209 company-years, 34 countries, 4 types, 30 interaction terms.
F = 6.20; extra adjusted R² from the interaction = +0.0145; permutation p = 0.2075. (Null F: median 4.99, 95th percentile 7.69)


In [6]:
# for comparison: the usual cluster-robust Wald test of the same interaction terms (known to be optimistic with 34 clusters)
m = fit_cluster("roa_pp ~ C(sector) * C(ctype) + size_rank + C(fiscal_year)", dd)
inter = [n for n in m.params.index if ":" in n]
try:
    w = m.wald_test(", ".join(f"{n} = 0" for n in inter), scalar=True)
    print(f"Cluster-robust Wald test of the {len(inter)} interaction terms: p = {float(w.pvalue):.4f} (optimistic; the permutation p above is the one to use)")
except Exception as e:
    print("Wald test not available:", str(e)[:80])

Cluster-robust Wald test of the 30 interaction terms: p = 0.0000 (optimistic; the permutation p above is the one to use)


## 4 · Robustness
The permutation test repeated with different numbers of types (3, 4, 5), without financials and real estate, with only sectors that have at least 25 company-years in 3+ countries per type, and for non-EU and EU subsets. 499 shuffles each.

In [7]:
def run(data, labels, B=499):
    global y, countries_, Sd, Yd, base_cols
    data = data.reset_index(drop=True)
    Sd = pd.get_dummies(data.sector, drop_first=True).astype(float).values
    Yd = pd.get_dummies(data.fiscal_year, drop_first=True).astype(float).values
    base_cols = np.column_stack([np.ones(len(data)), Sd, data.size_rank.values, Yd])
    y, countries_ = data.roa_pp.values, data.iso3.values
    used = pd.Series(labels).loc[sorted(set(countries_))]
    f0, r2, _ = fstat(used)
    r = np.random.default_rng(1)
    nul = [fstat(pd.Series(r.permutation(used.values), index=used.index))[0] for _ in range(B)]
    return f0, r2, (np.sum(np.array(nul) >= f0) + 1) / (B + 1)
rows = []
for k in (3, 4, 5):
    lk, ik = utils.country_clusters(profile, k_range=(k, k), min_size=1)
    f0, r2, p = run(d, lk); rows.append({"variant": f"{k} types", "F": f0, "extra adj. R²": r2, "p (permutation)": p, "countries": d.iso3.nunique()})
for name, sub, lbl in [("without financials & real estate", d[~d.sector.isin(["Financial Services", "Real Estate"])], lab),
                       ("EU countries only", d[d.iso3.isin(dim.index[dim.is_eu])], lab), ("non-EU countries only", d[~d.iso3.isin(dim.index[dim.is_eu])], lab)]:
    f0, r2, p = run(sub, lbl); rows.append({"variant": name, "F": f0, "extra adj. R²": r2, "p (permutation)": p, "countries": sub.iso3.nunique()})
rob = pd.DataFrame(rows).set_index("variant")
rob_share = float((rob["p (permutation)"] < 0.05).mean())
print(f"Interaction significant (p < 0.05) in {rob_share:.0%} of the {len(rob)} variants.")
rob

Interaction significant (p < 0.05) in 0% of the 6 variants.


,F,extra adj. R²,p (permutation),countries
variant,,,,
3 types,6.157,0.010,0.234,34
4 types,6.204,0.014,0.162,34
5 types,4.670,0.013,0.508,34
without financials & real estate,5.606,0.013,0.190,34
EU countries only,4.675,0.013,0.404,21
non-EU countries only,7.253,0.034,0.204,13


## 5 · Replication
ESEF has no sector field, so the sector × type test **cannot be replicated** on the independent source. This is a limit of the evidence, not a result.

## 6 · Verdict

In [8]:
v = utils.verdict(dR2, p_perm, 0, rob_share, int(dd.iso3.nunique()))
top = cell.assign(cell=cell.index.map(lambda t: f"{t[0]} / type {t[1]}")).sort_values("interaction")
lines = [
 f"Primary: sector × type interaction F = {F_obs:.2f}, extra adjusted R² = {dR2:+.4f}, permutation p = {p_perm:.4f} ({info['k']} types, {dd.iso3.nunique()} countries).",
 f"Robustness: significant in {rob_share:.0%} of {len(rob)} variants.",
 f"Largest departures (pp ROA, vs sector and type averages): {top.iloc[0]['cell']} {top.iloc[0]['interaction']:+.1f}, {top.iloc[-1]['cell']} {top.iloc[-1]['interaction']:+.1f}.",
 "Not replicable on ESEF (no sector field).",
 f"VERDICT H4: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H4", id="H4", title="Sector × country type", statement="The ROA differences between sectors depend on the type of country.",
                  expected_sign=0, effect=float(dR2), ci_low=None, ci_high=None, effect_unit="extra adjusted R² from sector × type interactions",
                  p_primary=p_perm, p_method="permutation of type labels across countries", n=int(len(dd)), clusters=int(dd.iso3.nunique()), robust_share=rob_share,
                  replication_coef=None, replication_p=None, replication_same_sign=None, verdict=v, notebook="1.4_sector_by_country_type.ipynb", notes=lines)

Primary: sector × type interaction F = 6.20, extra adjusted R² = +0.0145, permutation p = 0.2075 (4 types, 34 countries).
Robustness: significant in 0% of 6 variants.
Largest departures (pp ROA, vs sector and type averages): Energy / type 1 -2.1, Consumer Defensive / type 4 +1.9.
Not replicable on ESEF (no sector field).
VERDICT H4: NOT SUPPORTED


### Limits
- Country types are a statistical grouping of overlapping features; with 34 countries, some sector × type cells hold only a few countries, so single-cell results (the heatmaps) are descriptive and noisy.
- The cluster assignment is fixed from the country features, not tuned to ROA, so the test is not circular, but a different clustering would give different types (see the robustness variants).
- Not replicated on an independent source.
- Association, not causation.

In [9]:
con.close()